# AP01 - Daten einlesen

PostgreSQL-Datenbank im Terminal anschauen

Mit `pd.read_excel()` werden die Excel-Daten als DataFrames in Python eingelesen. Anschliessend schreibt `.to_sql()` diese DataFrames als Tabellen in die PostgreSQL-Datenbank `postgres`

Beispiel:

    df_bicycles.to_sql('bicycles', engine, if_exists='replace', index=False)

- `df_bicycles`: DataFrame mit den eingelesenen Fahrraddaten
- `'bicycles'`: Name der Tabelle, die in PostgreSQL erstellt wird
- `engine`: SQLAlchemy-Engine für den Datenbankzugriff
- `if_exists='replace'`: Eine bereits vorhandene Tabelle wird gelöscht und durch die Daten aus dem DataFrame ersetzt
- `index=False`: Der pandas-Zeilenindex wird nicht als zusätzliche Spalte gespeichert

**Im Codespace das Terminal öffnen und mit PostgreSQL verbinden:**

    docker exec -it postgres_container psql -U pgadmin -d postgres

| Bestandteil | Bedeutung |
|---|---|
| `docker exec -it` | Befehl interaktiv in einem laufenden Container ausführen |
| `postgres_container` | Name des PostgreSQL-Containers |
| `psql` | PostgreSQL-Kommandozeilenprogramm starten |
| `-U pgadmin` | Mit dem Benutzer pgadmin anmelden |
| `-d postgres` | Mit der Datenbank postgres verbinden |

**Wichtig:**
- PostgreSQL muss laufen und die gewünschten Tabellen müssen bereits geladen sein
- `bicycles` durch den Namen der gewünschten Tabelle ersetzen
- Die Datei `bike_shop.db` ist eine SQLite-Datenbank und wird nicht mit psql geöffnet

# AP01 - DB Navigation
**Tabellen und Daten anschauen:**

| Befehl | Bedeutung |
|---|---|
| `\dt` | Tabellen im aktuellen Suchpfad anzeigen, normalerweise im Schema public |
| `\dt public.*` | Alle Tabellen im Schema public anzeigen |
| `\d bicycles` | Spalten, Datentypen und Constraints der Tabelle bicycles anzeigen |
| `SELECT * FROM bicycles;` | Alle Zeilen und Spalten der Tabelle bicycles abrufen |
| `SELECT * FROM bicycles LIMIT 10;` | Maximal zehn Zeilen der Tabelle bicycles anzeigen; ohne ORDER BY ist die Auswahl nicht festgelegt |

**Navigation und Beenden:**

| Befehl | Bedeutung |
|---|---|
| `Ctrl + C` | Aktuelle Eingabe oder laufende Abfrage abbrechen |
| `\! clear` | Terminalanzeige leeren |
| `q` | Pager-Ansicht verlassen, wenn unten beispielsweise (END) steht |
| `\q` | psql beenden und zur Shell zurückkehren |

# AP01 – PostgreSQL
## Libraries und Arbeitsverzeichnis

- **pandas:** Dateien einlesen, Daten als DataFrames bearbeiten und SQL-Ergebnisse anzeigen
- **create_engine():** Verbindung zwischen Python und PostgreSQL verwalten
- **text():** SQL-Anweisungen für die Ausführung über eine SQLAlchemy-Verbindung vorbereiten
- **Path.cwd():** Aktuelles Arbeitsverzeichnis anzeigen. Relative Dateipfade gehen von diesem Ordner aus

- **Docker** stellt in deinem Codespace den PostgreSQL-Datenbankserver bereit, auf den dein Jupyter-Notebook zugreift. Excel ist eine Datei; PostgreSQL ist das Programm, das deine PostgreSQL-Abfragen ausführt. Dass die Excel-Datei im Repo liegt, macht ihre Tabellen noch nicht zu Datenbanktabellen. Mit `pd.read_excel()` liest du die Excel-Daten als DataFrame in Python ein und schreibst sie mit `.to_sql()` als Tabelle in PostgreSQL. Anschliessend führt PostgreSQL deine SQL-Abfrage aus, und `pd.read_sql_query()` lädt das Ergebnis wieder als DataFrame nach Python zurück

**Voraussetzung:** PostgreSQL läuft und der passende Python-Kernel ist ausgewählt

In [2]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# Current working directory
print('Arbeitsverzeichnis:', Path.cwd())

#NICHTS ANPASSEN: Imports und Path.cwd()
#ERKENNTNIS: Relative Dateipfade beziehen sich auf das Arbeitsverzeichnis des Notebooks

Arbeitsverzeichnis: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


## Datenbankverbindung erstellen und testen

Die Verbindungsdaten stammen aus der mitgelieferten docker-compose.yml.

- **user / password:** PostgreSQL-Benutzer und Passwort.
- **host:** Adresse des Datenbankservers; in diesem Repo gemäss README localhost.
- **port:** PostgreSQL-Port 5432.
- **database:** Name der Datenbank; hier postgres.
- **db_connection_url:** Verbindungsadresse aus diesen Angaben.
- **create_engine():** Erstellt die SQLAlchemy-Engine.
- **engine.connect():** Öffnet eine Verbindung.
- **SELECT current_database():** Gibt den Namen der verbundenen Datenbank zurück.
- **engine.dispose():** Gibt die vom Verbindungspool gehaltenen Verbindungen frei.

**Erwartetes Ergebnis:** postgres.

**Wichtig:** Der erfolgreiche Verbindungstest bedeutet noch nicht, dass die Bike-Shop- oder Cinema-Tabellen bereits geladen sind.

In [3]:
# Set up database connection
user = 'pgadmin'                                                                #Benutzername für PostgreSQL
password = 'geheim'                                                             #Passwort des Datenbankbenutzers
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers
port = '5432'                                                                   #Port, über den PostgreSQL erreichbar ist
database = 'postgres'                                                           #Name der Datenbank, mit der verbunden wird

# Create Connection URL
db_connection_url = (
    f'postgresql://{user}:{password}@{host}:{port}/{database}'                  #Verbindungsadresse aus den oben definierten Angaben zusammensetzen
)                                                                               

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine erstellen; Verbindung wird erst bei Bedarf geöffnet

# Test database connection
with engine.connect() as connection:                                            #Verbindung öffnen und nach dem Block automatisch schliessen
    result = connection.execute(text('SELECT current_database()'))              #Namen der aktuellen Datenbank abfragen
    print('Verbundene Datenbank:', result.scalar())                             #Einzelnen Ergebniswert auslesen und anzeigen

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

Verbundene Datenbank: postgres


## Tipps
#### display(df_bicycles) bspw. in einen neuen Codeblock eingeben, um den Dataframe anzuzeigen

In [9]:
display(df_bicycles)

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


### Retrieve all rows and all columns from the bicycles table

In [6]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine für die Datenbankverbindung erstellen

# Execute SQL query
df_bicycles = pd.read_sql_query("""
    SELECT *                                                --Alle Spalten auswählen
    FROM bicycles;                                          --Daten aus der Tabelle bicycles abrufen
""", con=engine)                                                                #SQL ausführen und Ergebnis als DataFrame speichern

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

# Show results
display(df_bicycles)                                                            #Ergebnistabelle anzeigen

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


### Retrieve all distinct bicycle makes from the bicycles table.

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT DISTINCT make        --Jede Fahrradmarke nur einmal ausgeben
    FROM bicycles;              --Daten aus der Tabelle bicycles abrufen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make
0,Santa Cruz
1,Giant
2,Specialized
3,Merida
4,Cannondale
5,Trek
6,Bianchi
7,Cervelo
8,Cube
9,Scott


### Retrieve the make, model and price of all bicycles. Rename the price column to list_price in the result

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price AS list_price  --Preisspalte im Ergebnis umbenennen; Originalspalte bleibt unverändert
    FROM bicycles;              --Daten aus der Tabelle bicycles abrufen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model,list_price
0,Trek,Domane,5066.94
1,Merida,Reacto,5529.38
2,Merida,Big Nine,1518.33
3,Trek,Marlin,5038.99
4,Merida,Scultura,1548.19
...,...,...,...
1995,Trek,Emonda,2376.02
1996,Merida,Scultura,2833.36
1997,Merida,Scultura,1629.91
1998,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all Bianchi bicycles

In [ ]:
engine = create_engine(db_connection_url)

df_result = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE make = 'Bianchi';     --Nur Fahrräder der Marke Bianchi auswählen; Textwerte in einfachen Anführungszeichen
""", con=engine)

engine.dispose()
display(df_result)

### Retrieve the bicycle ID, make, model and price of all bicycles

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Eindeutige Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen; kein Komma vor FROM
    FROM bicycles;              --Daten aus der Tabelle bicycles abrufen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all bicycles with a price greater than 3000

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE price > 3000;         --Nur Preise über 3000 auswählen; Zahlen ohne Anführungszeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,C2UQZP803C,Trek,Marlin,5038.99
3,C4RQCO25I9,Cube,Reaction,3339.27
4,C9LPTOS9KE,Cannondale,Synapse,5399.61
...,...,...,...,...
1389,CFFTG1MNW0,Bianchi,Aria,5939.92
1390,C6DHR3STN7,Trek,Marlin,6624.90
1391,CN5W80Q44T,Santa Cruz,Bronson,5442.78
1392,CX9QJEJHDE,Giant,Talon,5264.70


### Retrieve the bicycle ID, make, model and price of all Bianchi bicycles with a price greater than 3000

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE make = 'Bianchi'      --Erste Bedingung: Marke ist Bianchi (in Anführungszeichen)
      AND price > 3000;         --Zweite Bedingung: Preis über 3000; beide Bedingungen müssen erfüllt sein
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CA2FL7YUGR,Bianchi,Aria,6053.73
...,...,...,...,...
139,C46AFS3V3O,Bianchi,Aria,5965.55
140,C6FYCWDJKF,Bianchi,Infinito,6983.40
141,CVG2ERXCAM,Bianchi,Oltre,3068.96
142,CB4PK8PLSN,Bianchi,Oltre,7765.04


### Retrieve the bicycle ID, make, model and price of all bicycles that are either made by Bianchi or have a price below 1000

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE make = 'Bianchi'      --Erste Bedingung: Marke ist Bianchi
       OR price < 1000;         --Zweite Bedingung: Preis unter 1000; mindestens eine Bedingung muss erfüllt sein
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CCKYJCLZKE,Santa Cruz,Bronson,802.81
2,CJ3K49UQS0,Giant,TCR,918.55
3,CTFUCPJ6CQ,Bianchi,Aria,7050.68
4,CCP6K2ZAAU,Bianchi,Aria,3965.39
...,...,...,...,...
229,C49VNDE7VN,Giant,Defy,916.97
230,CLKN05TSC9,Specialized,Rockhopper,878.78
231,CB4PK8PLSN,Bianchi,Oltre,7765.04
232,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles that are not made by Bianchi

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,               --Fahrrad-ID auswählen
           make,                    --Fahrradmarke auswählen
           model,                   --Fahrradmodell auswählen
           price                    --Listenpreis auswählen
    FROM bicycles                   --Daten aus der Tabelle bicycles abrufen
    WHERE NOT make = 'Bianchi';     --Bianchi ausschliessen; entspricht make <> 'Bianchi'; NULL wird nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1804,CX9QJEJHDE,Giant,Talon,5264.70
1805,CLH919EIKV,Trek,Emonda,2376.02
1806,CP3F48BE9Y,Merida,Scultura,2833.36
1807,CHBXQ3SSNU,Merida,Scultura,1629.91


### Retrieve the bicycle ID, make, model and price of all bicycles with a price between 2000 and 4000, including both limits

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price BETWEEN 2000 AND 4000;      --Preise von 2000 bis 4000 auswählen; beide Grenzen sind eingeschlossen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C4RQCO25I9,Cube,Reaction,3339.27
1,CWGKVNNCYD,Giant,TCR,2087.55
2,CCI58KF4MJ,Cube,Stereo,3047.13
3,CNKHIRR7HH,Santa Cruz,Bronson,2314.02
4,C4ZC4AYR1E,Trek,Emonda,2198.88
...,...,...,...,...
536,CXSPK20YP0,Cervelo,R5,3719.35
537,CDBAWQPE4N,Cervelo,Soloist,2448.92
538,CLH919EIKV,Trek,Emonda,2376.02
539,CP3F48BE9Y,Merida,Scultura,2833.36


### Retrieve the bicycle ID, make, model and price of all bicycles with a price outside the range of 2000 to 4000

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                           --Fahrrad-ID auswählen
           make,                                --Fahrradmarke auswählen
           model,                               --Fahrradmodell auswählen
           price                                --Listenpreis auswählen
    FROM bicycles                               --Daten aus der Tabelle bicycles abrufen
    WHERE price NOT BETWEEN 2000 AND 4000;      --Preise unter 2000 oder über 4000 auswählen; Grenzen und NULL ausgeschlossen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1454,C6DHR3STN7,Trek,Marlin,6624.90
1455,CN5W80Q44T,Santa Cruz,Bronson,5442.78
1456,CX9QJEJHDE,Giant,Talon,5264.70
1457,CHBXQ3SSNU,Merida,Scultura,1629.91


### Retrieve the bicycle ID, make, model and price of all bicycles with a price of either 2087.55, 2943.58 or 2448.92

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                                --Fahrrad-ID auswählen
           make,                                     --Fahrradmarke auswählen
           model,                                    --Fahrradmodell auswählen
           price                                     --Listenpreis auswählen
    FROM bicycles                                    --Daten aus der Tabelle bicycles abrufen
    WHERE price IN (2087.55, 2943.58, 2448.92);      --Nur diese drei exakten Preise auswählen; entspricht drei mit OR verbundenen Vergleichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CWGKVNNCYD,Giant,TCR,2087.55
1,CDBAWQPE4N,Cervelo,Soloist,2448.92
2,CNJYADLN8R,Bianchi,Aria,2943.58


### Retrieve the bicycle ID, make, model and price of all bicycles whose price is not 2000, 3000 or 4000

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price NOT IN (2000, 3000, 4000);   --Diese drei exakten Preise ausschliessen; NULL-Preise werden ebenfalls nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all bicycles whose make starts with an uppercase B

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make LIKE 'B%%';                  --Beginnt mit B; Gross-/Kleinschreibung beachten; Platzhalter für psycopg2 verdoppeln
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles whose make starts with B or b, regardless of case

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make ILIKE 'b%%';                 --ILIKE ignoriert Gross-/Kleinschreibung; Platzhalter für psycopg2 verdoppeln
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles whose make contains the text "ian", regardless of case

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make ILIKE '%%ian%%';             --Text ian an beliebiger Stelle suchen; Gross-/Kleinschreibung ignorieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CWGKVNNCYD,Giant,TCR,2087.55
2,CJ3K49UQS0,Giant,TCR,918.55
3,CTFUCPJ6CQ,Bianchi,Aria,7050.68
4,CP64NZ2VJ0,Giant,Talon,4219.20
...,...,...,...,...
404,CB4PK8PLSN,Bianchi,Oltre,7765.04
405,CULZ43GQ7W,Giant,Defy,1759.41
406,CFFTG1MNW0,Bianchi,Aria,5939.92
407,CX9QJEJHDE,Giant,Talon,5264.70


### Retrieve the bicycle ID, make, model and price of all bicycles whose make ends with "chi", regardless of case

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make ILIKE '%%chi';               --Marke endet mit chi; davor beliebig viele Zeichen; Gross-/Kleinschreibung ignorieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles whose make does not start with B or b

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make NOT ILIKE 'b%%';             --Marken mit B oder b am Anfang ausschliessen; NULL wird ebenfalls nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1804,CX9QJEJHDE,Giant,Talon,5264.70
1805,CLH919EIKV,Trek,Emonda,2376.02
1806,CP3F48BE9Y,Merida,Scultura,2833.36
1807,CHBXQ3SSNU,Merida,Scultura,1629.91


### Retrieve the bicycle ID, make, model and price of all bicycles whose make has exactly seven characters and starts with an uppercase B

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make LIKE 'B______';              --B gefolgt von genau sechs Zeichen; jeder Unterstrich steht für genau ein Zeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles with a missing price

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NULL;                    --Fehlende Preise auswählen; IS NULL verwenden, niemals = NULL
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price


### Retrieve the bicycle ID, make, model and price of all bicycles with a known price

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NOT NULL;                --Nur vorhandene Preise auswählen; ein Preis von 0 bleibt enthalten
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all bicycles. Sort the result by price in ascending order, with missing prices last

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    ORDER BY price ASC NULLS LAST;          --Nach Preis aufsteigend sortieren; fehlende Preise zuletzt anzeigen, alternativ "DESC"
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CS2RTMPG37,Specialized,Allez,800.26
1,CCKYJCLZKE,Santa Cruz,Bronson,802.81
2,CZVM7LWXRP,Specialized,Rockhopper,809.00
3,CNX3HEE5GS,Merida,Big Nine,813.47
4,C7ZPXA6EXM,Cannondale,SuperSix,822.42
...,...,...,...,...
1995,CTVKH7S3PZ,Trek,Marlin,7972.58
1996,CLQCGBXCAG,Merida,Scultura,7972.88
1997,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
1998,CENB0TLQKE,Bianchi,Oltre,7989.74


### Retrieve the bicycle ID, make, model and price of all bicycles. Sort by make in ascending order and, within each make, by price in descending order, with missing prices last

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    ORDER BY make ASC,                      --Zuerst nach Marke alphabetisch aufsteigend sortieren
             price DESC NULLS LAST;         --Innerhalb derselben Marke nach Preis absteigend; fehlende Preise zuletzt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CENB0TLQKE,Bianchi,Oltre,7989.74
1,C7JUNJQSTR,Bianchi,Infinito,7963.11
2,C3MZ4T3WRF,Bianchi,Oltre,7951.93
3,CVSWF385E4,Bianchi,Oltre,7859.94
4,CV8A3G1LOS,Bianchi,Oltre,7824.00
...,...,...,...,...
1995,C1U71ESAXN,Trek,Emonda,1109.75
1996,CYEUN8XEM0,Trek,Domane,1091.15
1997,C32MK7YOH3,Trek,Domane,1074.92
1998,CYQC3N4R7I,Trek,Marlin,1054.77


### Retrieve the bicycle ID, make, model and price of the five most expensive bicycles with a known price. If prices are equal, sort by bicycle ID in ascending order

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NOT NULL                 --Fahrräder mit fehlendem Preis ausschliessen
    ORDER BY price DESC,                    --Teuerste Fahrräder zuerst anzeigen
             bicycleid ASC                  --Bei gleichem Preis nach Fahrrad-ID aufsteigend sortieren
    LIMIT 5;                                --Ausgabe auf maximal fünf Zeilen begrenzen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.90
1,CENB0TLQKE,Bianchi,Oltre,7989.74
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
3,CLQCGBXCAG,Merida,Scultura,7972.88
4,CTVKH7S3PZ,Trek,Marlin,7972.58


### Retrieve the bicycle ID, make, model and price of the sixth to tenth most expensive bicycles with a known price. If prices are equal, sort by bicycle ID in ascending order

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NOT NULL                 --Fahrräder mit fehlendem Preis ausschliessen
    ORDER BY price DESC,                    --Teuerste Fahrräder zuerst anzeigen
             bicycleid ASC                  --Bei gleichem Preis nach Fahrrad-ID aufsteigend sortieren
    LIMIT 5                                 --Maximal fünf Zeilen ausgeben
    OFFSET 5;                               --Die ersten fünf Zeilen der sortierten Ergebnismenge überspringen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CNS05492DI,Cube,Reaction,7969.30
1,CV56P1JD32,Scott,Addict,7968.66
2,CU0L6RJUXR,Specialized,Rockhopper,7964.52
3,C7JUNJQSTR,Bianchi,Infinito,7963.11
4,C1YLDNRGD4,Santa Cruz,Hightower,7962.93
